# 🏥 HealthBridge Medical Center
## Notebook 05 — SHAP Feature Importance & Business Simulation

**Prerequisites:** Run notebooks 01–04. Saved models must exist in `models/`.

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv').squeeze()

model = joblib.load('models/xgboost.pkl')  # update to best model
print('Loaded model and test data')

## SHAP Analysis

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', max_display=15, show=False)
plt.title('Top 15 Features — SHAP Importance', fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.title('SHAP Beeswarm — Feature Impact Direction', fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

top10 = pd.DataFrame({'Feature': X_test.columns,
                      'Mean |SHAP|': np.abs(shap_values).mean(axis=0)
                     }).sort_values('Mean |SHAP|', ascending=False).head(10).reset_index(drop=True)
top10.index += 1
print('Top 10 Predictors of 30-Day Readmission:')
print(top10.to_string())

## Risk Score Distribution by Tier

In [ ]:
y_prob = model.predict_proba(X_test)[:,1]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, name, color in [(0,'Not Readmitted','#2E75B6'),(1,'Readmitted <30','#C00000')]:
    subset = y_prob[y_test == label]
    axes[0].hist(subset, bins=40, alpha=0.65, label=f'{name} (n={len(subset):,})',
                 color=color, edgecolor='white')
axes[0].set_xlabel('Predicted Risk Score'); axes[0].set_ylabel('Patient Count')
axes[0].set_title('Risk Score Distribution by Outcome', fontweight='bold')
axes[0].legend()

tiers = {'Low (0–0.3)':(0.0,0.3),'Medium (0.3–0.6)':(0.3,0.6),'High (0.6+)':(0.6,1.0)}
rates, counts = [], []
for lo, hi in [v for v in tiers.values()]:
    mask = (y_prob>=lo)&(y_prob<hi)
    counts.append(mask.sum())
    rates.append(y_test[mask].mean()*100 if mask.sum()>0 else 0)
bars = axes[1].bar(list(tiers.keys()), rates,
                   color=['#1D7A4F','#F0A500','#C00000'], edgecolor='black', alpha=0.85)
axes[1].set_ylabel('Actual Readmission Rate (%)')
axes[1].set_title('Readmission Rate by Risk Tier', fontweight='bold')
for bar, c, r in zip(bars, counts, rates):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{r:.1f}%
(n={c:,})', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/risk_tiers.png', dpi=150, bbox_inches='tight')
plt.show()

## HealthBridge Business Impact Simulation

In [ ]:
y_prob = model.predict_proba(X_test)[:,1]
annual_discharges   = 12400
intervention_effect = 0.25      # 25% reduction in readmissions for flagged patients
current_rate        = 0.182
hrrp_baseline       = 2_400_000

thresh = np.percentile(y_prob, 80)
mask   = y_prob >= thresh
n_flagged    = mask.sum()
flagged_rate = y_test[mask].mean()

prevented       = int(annual_discharges * 0.20 * flagged_rate * intervention_effect)
new_rate        = current_rate - (prevented / annual_discharges)
penalty_saving  = hrrp_baseline * (current_rate - new_rate) / current_rate

print('='*55)
print('  HEALTHBRIDGE IMPACT SIMULATION')
print('='*55)
print(f'  Patients flagged (top 20%):       {n_flagged:,}')
print(f'  Readmission rate in flagged group: {flagged_rate*100:.1f}%')
print(f'  Readmissions prevented per year:  {prevented:,}')
print(f'  New projected readmission rate:   {new_rate*100:.1f}%')
print(f'  National average:                 15.5%')
print(f'  Estimated HRRP penalty reduction: ${penalty_saving:,.0f}')
print('='*55)